# 헬스케어 데이터 기반 투표 여부(voted) 예측
### 태스크: 이진분류 (voted=1 → 1, voted=2 → 0)
### 평가지표: ROC-AUC


실험내용: SkipConnection 적용,

In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import roc_auc_score

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cpu


In [2]:
# 데이터 로드
train = pd.read_csv('train.csv')
test  = pd.read_csv('test_x.csv')
sub   = pd.read_csv('sample_submission.csv')

print(f"Train: {train.shape} | Test: {test.shape}")

Train: (45532, 78) | Test: (11383, 77)


In [3]:
# voted: 1 = 투표함(양성), 2 = 투표안함(음성) → 0/1 이진화
train['target'] = (train['voted'] == 1).astype(int)
y = train['target'].values
print(f"클래스 분포 — 투표함(1): {y.sum():,} ({y.mean():.1%}) | "
        f"투표안함(0): {(1-y).sum():,} ({1-y.mean():.1%})")


클래스 분포 — 투표함(1): 20,634 (45.3%) | 투표안함(0): 24,898 (54.7%)


In [4]:
# 피처 엔지니어링 
def feature_engineering(df, target_enc_maps=None):   # ← target_enc_maps 파라미터로 받기
    df = df.copy()

    qa_cols = [c for c in df.columns if c.startswith('Q') and c.endswith('A')]
    qe_cols = [c for c in df.columns if c.startswith('Q') and c.endswith('E')]
    tp_cols = [f'tp0{i}' for i in range(1, 10)] + ['tp10']
    wf_cols = ['wf_01', 'wf_02', 'wf_03']
    wr_cols = [c for c in df.columns if c.startswith('wr_')]

    # --- 설문 응답(QA): 1~5 Likert ---
    df['qa_sum']     = df[qa_cols].sum(axis=1)
    df['qa_mean']    = df[qa_cols].mean(axis=1)
    df['qa_std']     = df[qa_cols].std(axis=1)
    df['qa_min']     = df[qa_cols].min(axis=1)
    df['qa_max']     = df[qa_cols].max(axis=1)
    df['qa_extreme'] = ((df[qa_cols] == 1) | (df[qa_cols] == 5)).sum(axis=1) / len(qa_cols)
    df['qa_neutral'] = (df[qa_cols] == 3).sum(axis=1) / len(qa_cols)

    # --- 응답시간(QE): 로그 변환으로 이상치 완화 ---
    for c in qe_cols:
        df[c] = np.log1p(df[c].clip(lower=0))
    df['qe_mean']  = df[qe_cols].mean(axis=1)
    df['qe_std']   = df[qe_cols].std(axis=1)
    df['qe_total'] = df[qe_cols].sum(axis=1)
    df['qe_fast_ratio'] = (df[qe_cols] < df[qe_cols].median().median()).sum(axis=1) / len(qe_cols)

    # --- Dark Triad (tp01~tp10, 0~7점) ---
    df['tp_sum']    = df[tp_cols].sum(axis=1)
    df['tp_mean']   = df[tp_cols].mean(axis=1)
    df['tp_std']    = df[tp_cols].std(axis=1)
    df['tp10_high'] = (df['tp10'] >= 5).astype(int)

    # --- wf / wr (이진 체크박스 그룹) ---
    df['wf_sum']  = df[wf_cols].sum(axis=1)
    df['wr_sum']  = df[wr_cols].sum(axis=1)
    df['wr_mean'] = df[wr_cols].mean(axis=1)

    # --- 범주형 인코딩 ---
    age_order = {'10s': 0, '20s': 1, '30s': 2, '40s': 3,
                    '50s': 4, '60s': 5, '+70s': 6}
    df['age_ord'] = df['age_group'].map(age_order)

    df['gender_bin'] = df['gender'].map({'Male': 1, 'Female': 0}).fillna(2).astype(int)

    for col in ['race', 'religion']:
        le = LabelEncoder()
        df[col + '_enc'] = le.fit_transform(df[col].astype(str))

    # --- 타깃 인코딩 (race, religion, age_group) ---
    # 파라미터로 받은 target_enc_maps 사용 (전역변수 참조 제거)
    if target_enc_maps is not None:
        for col, te_map in target_enc_maps.items():
            global_mean = te_map.get('__global__', 0.5)
            df[col + '_te'] = df[col].map(
                {k: v for k, v in te_map.items() if k != '__global__'}
            ).fillna(global_mean)
    return df


def make_target_enc_maps(df, target_col='target',
                            cols=('race', 'religion', 'age_group')):
    """전체 train으로 타깃 인코딩 맵 생성 (test 제출용)"""
    maps = {}
    global_mean = df[target_col].mean()
    for col in cols:
        te = df.groupby(col)[target_col].mean().to_dict()
        te['__global__'] = global_mean
        maps[col] = te
    return maps


def make_target_enc_maps_oof(df_tr, df_val, target_col='target',
                                cols=('race', 'religion', 'age_group'),
                                smoothing=10):
    """OOF(out-of-fold) 타깃 인코딩 — CV 내부에서 leak 방지용"""
    maps = {}
    global_mean = df_tr[target_col].mean()
    for col in cols:
        stats = df_tr.groupby(col)[target_col].agg(['mean', 'count'])
        smoothed = (stats['count'] * stats['mean'] + smoothing * global_mean) / \
                    (stats['count'] + smoothing)
        te = smoothed.to_dict()
        te['__global__'] = global_mean
        maps[col] = te
    return maps


# ── Fix: train 전체 기준 TE 맵을 먼저 생성한 뒤 feature_engineering에 전달 ──
target_enc_maps_full = make_target_enc_maps(train, 'target')

train_fe = feature_engineering(train, target_enc_maps=target_enc_maps_full)
test_fe  = feature_engineering(test,  target_enc_maps=target_enc_maps_full)

# 학습에 사용할 피처 선택
DROP = ['index', 'voted', 'target', 'age_group', 'gender', 'race', 'religion']
feature_cols = [c for c in train_fe.columns if c not in DROP]
print(f"최종 피처 수: {len(feature_cols)}")

X      = train_fe[feature_cols].values.astype(np.float32)
X_test = test_fe[feature_cols].values.astype(np.float32)


최종 피처 수: 97


In [5]:
# 스케일링
scaler      = StandardScaler()
X_scaled    = scaler.fit_transform(X)
X_test_sc   = scaler.transform(X_test)


In [6]:
class ResidualBlock(nn.Module):
    def __init__(self, dim, dropout=0.3):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
        )
        # 잔차 합산 후 정규화
        self.norm = nn.BatchNorm1d(dim)
        self.act  = nn.ReLU()
        self.drop = nn.Dropout(dropout * 0.5)

    def forward(self, x):
        return self.drop(self.act(self.norm(x + self.block(x))))

class ResidualMLP(nn.Module):
    """
    Projection -> Residual Blocks × N -> Output
    - Projection: 입력 차원을 hidden_dim으로 맞춤
    - Residual Blocks: 동일 차원 유지하며 표현 강화
    - 단순 MLP 대비 같은 파라미터 수로 더 깊은 표현 가능
    """
    def __init__(self, input_dim, hidden_dim=256, n_blocks=3, dropout=0.3):
        super().__init__()
        # 입력 → hidden_dim 투영
        self.proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        # Residual 블록들
        self.blocks = nn.ModuleList(
            [ResidualBlock(hidden_dim, dropout) for _ in range(n_blocks)]
        )
        # 출력 레이어
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, x):
        x = self.proj(x)
        for block in self.blocks:
            x = block(x)
        return self.head(x).squeeze(1)


In [7]:
# 학습,평가 함수
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        # Gradient clipping: 폭발적 기울기 방지
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total += loss.item() * len(xb)
    return total / len(loader.dataset)


@torch.no_grad()
def predict(model, loader):
    model.eval()
    preds, labels = [], []
    for xb, yb in loader:
        prob = torch.sigmoid(model(xb.to(device))).cpu().numpy()
        preds.extend(prob)
        labels.extend(yb.numpy())
    return np.array(preds), np.array(labels)



In [8]:
# K-Fold 교차검증 + 앙상블 (ResidualMLP)
N_FOLDS    = 5
EPOCHS     = 100
BATCH_SIZE = 512
LR         = 1e-3
PATIENCE   = 15   # Early stopping patience

skf        = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_preds  = np.zeros(len(X_scaled))
test_preds = np.zeros(len(X_test_sc))
fold_aucs  = []

input_dim = X_scaled.shape[1]   # ← Fix: 루프 밖으로 이동 (모델 생성 전에 정의)

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_scaled, y)):
    print(f"\n{'─'*50}")
    print(f"  Fold {fold+1}/{N_FOLDS}")

    # DataLoader 구성
    def make_loader(X_, y_, shuffle):
        ds = TensorDataset(torch.FloatTensor(X_), torch.FloatTensor(y_))
        return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                            drop_last=shuffle)

    tr_loader  = make_loader(X_scaled[tr_idx],  y[tr_idx].astype(np.float32), True)
    val_loader = make_loader(X_scaled[val_idx],  y[val_idx].astype(np.float32), False)
    te_loader  = make_loader(X_test_sc, np.zeros(len(X_test_sc), dtype=np.float32), False)

    # 모델 초기화 (ResidualMLP)
    model     = ResidualMLP(input_dim).to(device)   # ← input_dim이 이미 정의된 상태
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
    criterion = nn.BCEWithLogitsLoss()

    best_auc, best_val_pred, no_improve = 0, None, 0

    for epoch in range(1, EPOCHS + 1):
        tr_loss = train_epoch(model, tr_loader, optimizer, criterion)
        val_pred, val_true = predict(model, val_loader)
        val_auc = roc_auc_score(val_true, val_pred)
        scheduler.step()

        if val_auc > best_auc:
            best_auc, best_val_pred = val_auc, val_pred.copy()
            torch.save(model.state_dict(), f'/tmp/fold{fold}_best.pt')
            no_improve = 0
        else:
            no_improve += 1

        if epoch % 20 == 0:
            print(f"    epoch {epoch:3d} | loss={tr_loss:.4f} | auc={val_auc:.4f} | best={best_auc:.4f}")

        if no_improve >= PATIENCE:
            print(f"    → Early stopping at epoch {epoch} (best={best_auc:.4f})")
            break

    # 베스트 모델로 테스트 예측
    model.load_state_dict(torch.load(f'/tmp/fold{fold}_best.pt', map_location=device))
    te_pred, _ = predict(model, te_loader)

    oof_preds[val_idx] = best_val_pred
    test_preds         += te_pred / N_FOLDS
    fold_aucs.append(best_auc)
    print(f"  Fold {fold+1} Best AUC: {best_auc:.4f}")

print(f"\n{'='*50}")
oof_auc = roc_auc_score(y, oof_preds)
print(f"  OOF AUC : {oof_auc:.4f}")
print(f"  Fold AUC: {[round(a, 4) for a in fold_aucs]}")
print(f"  Mean±Std: {np.mean(fold_aucs):.4f} ± {np.std(fold_aucs):.4f}")



──────────────────────────────────────────────────
  Fold 1/5
    epoch  20 | loss=0.5178 | auc=0.7649 | best=0.7753
    → Early stopping at epoch 23 (best=0.7753)
  Fold 1 Best AUC: 0.7753

──────────────────────────────────────────────────
  Fold 2/5
    epoch  20 | loss=0.5157 | auc=0.7554 | best=0.7667
    → Early stopping at epoch 22 (best=0.7667)
  Fold 2 Best AUC: 0.7667

──────────────────────────────────────────────────
  Fold 3/5
    → Early stopping at epoch 18 (best=0.7561)
  Fold 3 Best AUC: 0.7561

──────────────────────────────────────────────────
  Fold 4/5
    epoch  20 | loss=0.5146 | auc=0.7457 | best=0.7553
    → Early stopping at epoch 21 (best=0.7553)
  Fold 4 Best AUC: 0.7553

──────────────────────────────────────────────────
  Fold 5/5
    epoch  20 | loss=0.5180 | auc=0.7593 | best=0.7617
    → Early stopping at epoch 26 (best=0.7617)
  Fold 5 Best AUC: 0.7617

  OOF AUC : 0.7628
  Fold AUC: [0.7753, 0.7667, 0.7561, 0.7553, 0.7617]
  Mean±Std: 0.7630 ± 0.0074

In [ ]:
# sample_submission의 voted 컬럼에 양성 클래스(투표함) 확률 기입
sub['voted'] = 1 - test_preds
sub.to_csv('submission03.csv', index=False)
print("\n 제출 파일 저장: submission03.csv")
print(sub.head())


 제출 파일 저장: submission02.csv
   index     voted
0      0  0.758701
1      1  0.891457
2      2  0.404745
3      3  0.241967
4      4  0.836315
